# 04.1 · q-transfer (Spec-1) — data pipeline demo

`SRC.task == DST.task`, `SRC.progress != DST.progress`. One multi-predicate task; a different subset is prefilled as done in SRC vs DST, so at the shared pivot they are about to write different next sub-goals. Transplant the SRC pivot **query** into DST → does DST now pursue SRC's next sub-goal (`expected_next`)?

This notebook demonstrates `src.diff.data.q_transfer_setups` + the query-inject machinery from `src.diff.functional`.

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0, 2")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

sys.path.append("../..")  # repo root, so `src` is importable
import torch, numpy as np
import src.diff.functional as P
from src.diff.functional import *          # helpers into scope
import src.diff.data as D                   # dataset generators

P.setup("Qwen/Qwen2.5-Coder-32B-Instruct") # loads model + head masks (once)
mt = P.mt
GOAL_HEADS, BOUNDARY_HEADS, UNION_HEADS, GROUP = P.GOAL_HEADS, P.BOUNDARY_HEADS, P.UNION_HEADS, P.GROUP
PATCH_HEADS = GOAL_HEADS                    # heads carrying the transplant (GOAL | UNION | ...)
from rich.panel import Panel
from rich.text import Text
from rich.markup import escape
from src.utils.experiment_utils import set_seed

def pivot_of(prompt, neg_shift=-1):
    return prepare_input(prompts=prompt, tokenizer=mt)["input_ids"].shape[1] + neg_shift

def show(prompt, title, emph_span=None, piv=-1):
    kw = dict(patch_pos=[piv])
    if emph_span is not None:
        kw["emphasize_tokens"] = range(*emph_span)
    rich_print(Panel(Text.from_markup(exact_prompt(prompt, **kw)),
                     title=f"[bold]{title}[/bold]", title_align="left",
                     border_style="grey42", padding=(1, 1)))

2026-09-07 18:27:42 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

Qwen/Qwen2.5-Coder-32B-Instruct
n_heads=40  n_kv_heads=8  group=5  head_dim=128  n_layer=64
GOAL_HEADS     : 122  (layers 28–55)
BOUNDARY_HEADS : 140  (layers 11–62)
UNION_HEADS    : 228


## Inspect the generated contrastive setups

In [2]:
# a few contrastive q-transfer setups (one datatype, listed presentation, python)
setups = D.q_transfer_setups("list[str]", presentation="listed", src_lang="python", dst_lang="python")
print(f"{len(setups)} setups for list[str]:")
for s in setups:
    print(f"  expected_next={s.expected_next:<18} src_progress={s.src_progress}  dst_progress={s.dst_progress}")

# datatype x presentation x language coverage (string render only — no model needed)
for dt in D.QTASKS:
    for pres in ("listed", "description"):
        n = len(D.q_transfer_setups(dt, presentation=pres))
        print(f"  {dt:<10} {pres:<12} -> {n} setups")

8 setups for list[str]:
  expected_next=all_unique_lengths src_progress=['count_starts_vowel', 'has_palindrome', 'longest_word_length']  dst_progress=['count_starts_vowel']
  expected_next=any_empty          src_progress=['all_unique_lengths', 'count_ends_consonant', 'distinct_count']  dst_progress=['all_unique_lengths']
  expected_next=total_vowels       src_progress=['count_capitalized', 'longest_word_length', 'count_short']  dst_progress=['count_capitalized']
  expected_next=longest_word_length src_progress=['count_ends_consonant', 'count_long']  dst_progress=['count_ends_consonant']
  expected_next=count_short        src_progress=['longest_word_length', 'count_ends_consonant']  dst_progress=['longest_word_length']
  expected_next=distinct_count     src_progress=['total_length', 'total_uppercase_letters', 'any_empty', 'count_ends_consonant']  dst_progress=['total_length']
  expected_next=shortest_word_length src_progress=['longest_word_length', 'count_all_lowercase', 'total_length',

## Render one setup — SRC and DST prompts with pivots

In [3]:
set_seed(123)
inst = D.q_transfer_setups(
    "list[str]", 
    # presentation="listed", 
    src_lang="python", dst_lang="python", 
    n_conditions=3,
    naming="letters"
)
print(len(inst))

inst = inst[3]  # pick one setup to visualize
m = D.materialize(inst)                       # applies the chat template
src_prompt, dst_prompt = m["src_prompt"], m["dst_prompt"]
piv_src, piv_dst = pivot_of(src_prompt), pivot_of(dst_prompt)
print(f"expected_next = {inst.expected_next!r}  ({inst.expected_next_desc})")
show(src_prompt, f"SRC · progress={inst.src_progress} · about to write {inst.expected_next!r} · pivot@{piv_src}", piv=-1)
show(dst_prompt, f"DST · progress={inst.dst_progress} · pivot@{piv_dst}", piv=-1)

8
expected_next = 'distinct_count'  (the number of distinct words)


╭─ SRC · progress=['count_long', 'total_uppercase_letters'] · about to write 'distinct_count' · pivot@149 ────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the number of words longer than five characters\n                                                           │
│ (2) the total number of uppercase letters across all words\n                                                    │
│ (3) the number of distinct words\n                                                                              │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = sum(1 for w in words if len(w) > 5)\n                                                                   │
│     b = sum(1 for w in words for c in w if c.isupper())\n                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ DST · progress=['count_long'] · pivot@130 ─────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the number of words longer than five characters\n                                                           │
│ (2) the total number of uppercase letters across all words\n                                                    │
│ (3) the number of distinct words\n                                                                              │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = sum(1 for w in words if len(w) > 5)\n                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Intervention — transplant the SRC pivot query into DST

Cache the (pre-RoPE) query of `PATCH_HEADS` at the SRC pivot; inject it at the DST pivot (prefill + re-clamped for `n_steer` decode steps). Expect the patched DST to implement `expected_next` instead of its own next predicate.

In [4]:
# ── intervention: transplant the SRC pivot query into DST — grouped per run ──
from src.attention import get_attention_matrices
Q_LAYERS = sorted({l for l, _ in PATCH_HEADS})
q_src = cache_src_proj(src_prompt, [piv_src], Q_LAYERS, "q")

def q_inject_specs(qcache, dest_pivot, heads=PATCH_HEADS):
    qh = heads_per_layer(heads, "q")
    return ([dict(which="q", cache=qcache, heads=qh, positions=[dest_pivot])],   # prefill
            [dict(which="q", cache=qcache, heads=qh, positions=[-1])])            # decode re-clamp

src_tok = prepare_input(prompts=src_prompt, tokenizer=mt)
dst_tok = prepare_input(prompts=dst_prompt, tokenizer=mt)
pf, dc  = q_inject_specs(q_src, piv_dst)

# clean baseline attn over the TASK (numbered predicate list); query = pivot (last token)
_spans    = [token_span(dst_prompt, p.description) for p in inst.task.predicates]
TASK_POS  = list(range(min(s[0] for s in _spans), max(s[1] for s in _spans)))
task_toks = [mt.tokenizer.decode([int(dst_tok["input_ids"][0][j])]) for j in TASK_POS]
_clean_i  = get_attention_matrices(dst_tok, mt, value_weighted=True)
def _cattn(h): return pivot_attn(_clean_i, list(h), TASK_POS, piv_dst)

# ── source_run — clean SRC (frontier = expected_next); generation + goalscope ──
gen_src = greedy(src_tok, max_new=96)
rich_print(f"[red1]source_run · clean SRC (frontier {inst.expected_next!r}):[/red1] \n{escape(gen_src)}")
ro_src = goalscope_readout("   ↳ goalscope · source_run", src_prompt, piv_src, heads=PATCH_HEADS)[0]["patched_gen"]

# ── destination_run — clean DST (frontier = dst_next); generation + goalscope + ATTN_VIS baseline ──
gen_dst = greedy(dst_tok, max_new=96)
rich_print(f"[sea_green3]destination_run · clean DST (frontier {inst.meta['dst_next']!r}):[/sea_green3] \n{escape(gen_dst)}")
ro_dst = goalscope_readout("   ↳ goalscope · destination_run", dst_prompt, piv_dst, heads=PATCH_HEADS)[0]["patched_gen"]
attn_clean_goal = _cattn(GOAL_HEADS)      # clean baseline for the gaze eval
show_delta_tokens(task_toks, _cattn(PATCH_HEADS), "ATTN_VIS · pivot→TASK (clean · PATCH_HEADS)")
# show_delta_tokens(task_toks, _cattn(GOAL_HEADS),  "ATTN_VIS · pivot→TASK (clean · GOAL_HEADS)")

# ── patched_run — SRC query injected into DST (expect expected_next); gen + goalscope + ATTN_DIFF ──
_iface = make_patch_interface(mt, pf)
gen_patched = greedy(dst_tok, prefill_specs=pf, decode_specs=dc, n_steer=32, max_new=96)
rich_print(f"[salmon1]patched_run · DST (expect {inst.expected_next!r}):[/salmon1] \n{escape(gen_patched)}")
ro_patched = goalscope_readout("   ↳ goalscope · patched_run", dst_prompt, piv_dst, heads=PATCH_HEADS, iface=_iface)[0]["patched_gen"]
_patched_i = get_attention_matrices(dst_tok, mt, value_weighted=True, patch_interface=_iface)
attn_patched_goal = pivot_attn(_patched_i, list(GOAL_HEADS), TASK_POS, piv_dst)   # patched, for gaze
show_delta_tokens(task_toks, pivot_attn(_patched_i, list(PATCH_HEADS), TASK_POS, piv_dst) - _cattn(PATCH_HEADS),
                  "ATTN_DIFF · patched−clean (PATCH_HEADS)")
show_delta_tokens(task_toks, attn_patched_goal, "ATTN_VIS · patched (GOAL_HEADS)")

source_run · clean SRC (frontier 'distinct_count'): 
 c = len(set(words))
    return a, b, c
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


   ↳ goalscope · source_run  pos=149 '    ' => Identifying unique elements in a list.

destination_run · clean DST (frontier 'total_uppercase_letters'): 
 b = sum(c.isupper() for w in words for c in w)
    c = len(set(words))
    return a, b, c
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

... (3 outputs left out)

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · patched_run  pos=130 '    ' => Identifying unique elements in a list.

ATTN_DIFF · patched−clean (PATCH_HEADS)   (green +Δ / red −Δ, |Δ|max=0.956)

ATTN_VIS · patched (GOAL_HEADS)   (green +Δ / red −Δ, |Δ|max=0.723)

---
## Evaluate the intervention (`src.diff.eval`)

Three complementary evals of the q-transfer intervention (single source of truth in `src/diff/eval.py`):

1. **Causal** (what we care about) — oracle judges whether the patched continuation pursues the SRC target (or a non-trivial part).
2. **Readout shift** — classifies all three goalscope readouts (source/destination/patched); a case is *valid* only if the clean readouts land as expected (source→SRC, dest→DST), and *shifted* if the patched readout expresses the SRC target.
3. **Gaze** — rule-based: did the value-weighted attention delta move toward the SRC-target tokens vs the DST-target tokens?

The causal & readout evals call the Claude oracle (`ask_claude`, disk-cached). The same `eval_q_transfer` works on the composed-filter instance below.

In [5]:
from src.diff import eval as E

qeval = E.eval_q_transfer(
    inst,
    patched_gen=gen_patched,                                   # 1. causal (oracle)
    readouts=(ro_src, ro_dst, ro_patched),                    # 2. readout shift (oracle)
    gaze=dict(prompt=dst_prompt, task_positions=TASK_POS,     # 3. gaze (rule-based, absolute)
              attn_patched=attn_patched_goal, attn_clean=attn_clean_goal),
)
rich_print(f"[bold]EVAL[/bold] · expected_next={inst.expected_next!r}\n{qeval.summary()}")
rich_print("  causal :", qeval.causal)
rich_print("  readout:", qeval.readout)
rich_print("  gaze   :", qeval.gaze)

EVAL · expected_next='distinct_count'
causal: TRIGGERED (src) | readout: src→dst→src  | gaze: src+0.314 vs dst+0.049 (shifted; clean_prefers_src=False)

causal :
{
    'label': 'src',
    'why': 'The first computation is `b = len(set(words))`, which counts distinct words by converting the list to a
set and taking its length — this directly pursues sub-goal (A).',
    'raw': '<verdict>A</verdict>\n<why>The first computation is `b = len(set(words))`, which counts distinct words 
by converting the list to a set and taking its length — this directly pursues sub-goal (A).</why>',
    'triggered': True
}

readout:
{
    'source': 'src',
    'dest': 'dst',
    'patched': 'src',
    'valid': True,
    'shifted': True,
    'details': {
        'source': {
            'label': 'src',
            'why': 'Identifying unique elements in a list corresponds to finding distinct words, which is sub-goal 
A.',
            'raw': '<verdict>A</verdict><why>Identifying unique elements in a list corresponds to finding distinct 
words, which is sub-goal A.</why>'
        },
        'dest': {
            'label': 'dst',
            'why': 'The phrase explicitly describes counting uppercase letters, which directly corresponds to 
sub-goal B (total number of uppercase letters across all words).',
            'raw': '<verdict>B</verdict><why>The phrase explicitly describes counting uppercase letters, which 
directly corresponds to sub-goal B (total number of uppercase letters across all words).</why>'
        },
        'patched': {
            'label': 'src',
            'why': 'Identifying unique elements in a list corresponds to finding distinct words, which is sub-goal 
A.',
            'raw': '<verdict>A</verdict><why>Identifying unique elements in a list corresponds to finding distinct 
words, which is sub-goal A.</why>'
        }
    }
}

gaze   :
{
    'src_patched': 0.3139765776693821,
    'dst_patched': 0.04890604493104749,
    'shifted': True,
    'margin': 0.2650705327383346,
    'src_clean': 0.05832146145403385,
    'dst_clean': 0.2653054628107283,
    'clean_prefers_src': False
}

---
## Composed-filter variant (`filter_transfer_setups`) — mid-expression pivot

Same query-transfer *mechanism*, but the sub-goals are **commutative conditions composed inside one return expression** (the original 00_diff TASK_1 framing: `return all the words that are palindromes and start with a vowel`). The pivot sits **mid-expression** — right after `… if <cond> and` — so the model completes the *same* comprehension by adding the next condition, rather than opening a fresh assignment statement.

The intervention is unchanged (reuses `q_inject_specs`); only the ATTN **TASK span** is located differently — the composed phrase `m["task_text"]`, not a per-predicate numbered list (`FilterTask` has `.conditions`, not `.predicates`).

In [6]:
# composed-filter setups: 2 commutative sub-tasks; SRC/DST prefill DIFFERENT one (00_diff TASK_1).
fsetups = D.filter_transfer_setups("list[str]")          # postfix="targeted" by default
print(f"{len(fsetups)} filter setups for list[str] — TASK: {fsetups[0].task_text!r}")
for s in fsetups:
    print(f"  src_fill={s.src_fill:<12} dst_fill={s.dst_fill:<12} -> expected_next={s.expected_next!r} "
          f"(clean DST -> {s.meta['dst_next']!r})")
print()
for dt in D.FILTER_TASKS:
    print(f"  {dt:<10} -> {len(D.filter_transfer_setups(dt))} setups ({len(D.FILTER_TASKS[dt].conditions)} sub-tasks in bank)")

8 filter setups for list[str] — TASK: 'all the words in `words` that start with the letter a and are unchanged when reversed'
  src_fill=same_reversed dst_fill=starts_a     -> expected_next='starts_a' (clean DST -> 'same_reversed')
  src_fill=all_lower    dst_fill=vowel_start  -> expected_next='vowel_start' (clean DST -> 'all_lower')
  src_fill=short        dst_fill=same_reversed -> expected_next='same_reversed' (clean DST -> 'short')
  src_fill=all_alpha    dst_fill=length_five  -> expected_next='length_five' (clean DST -> 'all_alpha')
  src_fill=short        dst_fill=contains_digit -> expected_next='contains_digit' (clean DST -> 'short')
  src_fill=all_lower    dst_fill=starts_a     -> expected_next='starts_a' (clean DST -> 'all_lower')
  src_fill=all_lower    dst_fill=same_reversed -> expected_next='same_reversed' (clean DST -> 'all_lower')
  src_fill=last_is_digit dst_fill=long         -> expected_next='long' (clean DST -> 'last_is_digit')

  list[str]  -> 8 setups (16 sub-tasks in

In [7]:
finst = D.filter_transfer_setups(
    "list[str]",
    src_lang="python", dst_lang="python",
    postfix="targeted"
)[0]          # or postfix="return" for the loose pivot
fm = D.materialize(finst)
# fsrc, fdst = fm["src_prompt"].rstrip(), fm["dst_prompt"].rstrip()
fsrc, fdst = fm["src_prompt"], fm["dst_prompt"]
fpiv_src, fpiv_dst = pivot_of(fsrc), pivot_of(fdst)
print(f"expected_next = {finst.expected_next!r}  ({finst.expected_next_desc})")
show(fsrc, f"SRC · prefilled {finst.src_fill!r} · about to apply {finst.expected_next!r} · pivot@{fpiv_src}",
     emph_span=token_span(fsrc, fm["task_text"]), piv=-1)
show(fdst, f"DST · prefilled {finst.dst_fill!r} · about to apply {finst.meta['dst_next']!r} · pivot@{fpiv_dst}",
     emph_span=token_span(fdst, fm["task_text"]), piv=-1)

expected_next = 'starts_a'  (start with the letter a)


╭─ SRC · prefilled 'same_reversed' · about to apply 'starts_a' · pivot@121 ───────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ all the words in `words` that start with the letter a and are unchanged when reversed\n                         │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     res = [w for w in words if w == w[::-1]]\n                                                                  │
│     return [w for w in res if                                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ DST · prefilled 'starts_a' · about to apply 'same_reversed' · pivot@120 ───────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ all the words in `words` that start with the letter a and are unchanged when reversed\n                         │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     res = [w for w in words if w.startswith('a')]\n                                                             │
│     return [w for w in res if                                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### Intervention — same query transplant, TASK span = the composed phrase

Reuses `q_inject_specs` / `Q_LAYERS` from the q-transfer intervention above; the only change is `TASK_POS = token_span(fdst, m["task_text"])`.

In [8]:
PATCH_HEADS = GOAL_HEADS  # heads carrying the transplant (GOAL | UNION | ...)
len(PATCH_HEADS)

122

In [9]:
# ── composed-filter intervention: transplant SRC pivot query into DST — grouped per run ──
from src.attention import get_attention_matrices
Q_LAYERS  = sorted({l for l, _ in PATCH_HEADS})
fq_src    = cache_src_proj(fsrc, [fpiv_src], Q_LAYERS, "q")
fsrc_tok  = prepare_input(prompts=fsrc, tokenizer=mt)
fdst_tok  = prepare_input(prompts=fdst, tokenizer=mt)
fpf, fdc  = q_inject_specs(fq_src, fpiv_dst)              # SAME machinery as the q-transfer cell

# TASK span = the single COMPOSED phrase (FilterTask has .conditions, not a per-line list); query = pivot
TASK_POS  = list(range(*token_span(fdst, fm["task_text"])))
task_toks = [mt.tokenizer.decode([int(fdst_tok["input_ids"][0][j])]) for j in TASK_POS]
_clean_i  = get_attention_matrices(fdst_tok, mt, value_weighted=True)
def _cattn(h): return pivot_attn(_clean_i, list(h), TASK_POS, fpiv_dst)

# ── source_run — clean SRC (frontier = expected_next) ──
fgen_src = greedy(fsrc_tok, max_new=64)
rich_print(f"[red1]source_run · clean SRC (frontier {finst.expected_next!r}):[/red1] \n{escape(fgen_src)}")
fro_src = goalscope_readout("   ↳ goalscope · source_run", fsrc, fpiv_src, heads=PATCH_HEADS)[0]["patched_gen"]

# ── destination_run — clean DST (frontier = dst_next); ATTN_VIS baseline ──
fgen_dst = greedy(fdst_tok, max_new=64)
rich_print(f"[sea_green3]destination_run · clean DST (frontier {finst.meta['dst_next']!r}):[/sea_green3] \n{escape(fgen_dst)}")
fro_dst = goalscope_readout("   ↳ goalscope · destination_run", fdst, fpiv_dst, heads=PATCH_HEADS)[0]["patched_gen"]
fattn_clean_goal = _cattn(GOAL_HEADS)      # clean baseline for the gaze eval
show_delta_tokens(task_toks, _cattn(PATCH_HEADS), "ATTN_VIS · pivot→TASK (clean · PATCH_HEADS)")
# show_delta_tokens(task_toks, _cattn(GOAL_HEADS),  "ATTN_VIS · pivot→TASK (clean · GOAL_HEADS)")

# ── patched_run — SRC query injected into DST (expect expected_next); ATTN_DIFF ──
_iface = make_patch_interface(mt, fpf)
fgen_patched = greedy(fdst_tok, prefill_specs=fpf, decode_specs=fdc, n_steer=32, max_new=64)
rich_print(f"[salmon1]patched_run · DST (expect {finst.expected_next!r}):[/salmon1] \n{escape(fgen_patched)}")
fro_patched = goalscope_readout("   ↳ goalscope · patched_run", fdst, fpiv_dst, heads=PATCH_HEADS, iface=_iface)[0]["patched_gen"]
_patched_i = get_attention_matrices(fdst_tok, mt, value_weighted=True, patch_interface=_iface)
fattn_patched_goal = pivot_attn(_patched_i, list(GOAL_HEADS), TASK_POS, fpiv_dst)   # patched, for gaze
show_delta_tokens(task_toks, pivot_attn(_patched_i, list(PATCH_HEADS), TASK_POS, fpiv_dst) - _cattn(PATCH_HEADS),
                  "ATTN_DIFF · patched−clean (PATCH_HEADS)")
show_delta_tokens(task_toks, fattn_patched_goal, "ATTN_VIS · patched (GOAL_HEADS)")

source_run · clean SRC (frontier 'starts_a'): 
 w.startswith('a')]

# Example usage:
# words = ["abba", "abc", "a", "racecar", "level", "deified", "rotor", "ala"]
# print(func(words))  # Output: ['abba', 'a', 'ala']
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · source_run  pos=121 ' if' => I'm checking if the string starts with 'a'.

destination_run · clean DST (frontier 'same_reversed'): 
 w == w[::-1]]
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · destination_run  pos=120 ' if' => I'm checking if the string is equal to its reverse.

... (2 outputs left out)

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · patched_run  pos=120 ' if' => I'm checking if the string starts with 'a'.

ATTN_DIFF · patched−clean (PATCH_HEADS)   (green +Δ / red −Δ, |Δ|max=1.772)

ATTN_VIS · patched (GOAL_HEADS)   (green +Δ / red −Δ, |Δ|max=0.645)

### Evaluate the composed-filter intervention

Same `eval_q_transfer` (the filter instance exposes the same `expected_next_desc` / `dst_next_desc` / `task_desc`), on the filter run's captured outputs.

In [10]:
feval = E.eval_q_transfer(
    finst,
    patched_gen=fgen_patched,                                    # 1. causal (oracle)
    readouts=(fro_src, fro_dst, fro_patched),                    # 2. readout shift (oracle)
    gaze=dict(prompt=fdst, task_positions=TASK_POS,              # 3. gaze (rule-based, absolute)
              attn_patched=fattn_patched_goal, attn_clean=fattn_clean_goal),
)
rich_print(f"[bold]EVAL (filter)[/bold] · expected_next={finst.expected_next!r}\n{feval.summary()}")
rich_print("  causal :", feval.causal)
rich_print("  readout:", feval.readout)
rich_print("  gaze   :", feval.gaze)

EVAL (filter) · expected_next='starts_a'
causal: TRIGGERED (src) | readout: src→dst→src  | gaze: src+0.290 vs dst+0.141 (shifted; clean_prefers_src=False)

causal :
{
    'label': 'src',
    'why': "The condition `w[0] == 'a'` checks if the word starts with 'a' (sub-goal A); the `w == w.lower()` part 
checks case equality, not palindrome, so sub-goal B is not pursued.",
    'raw': "The first computation is `w == w.lower()` which checks if a word equals its lowercase version — this is
not checking palindrome (unchanged when reversed) nor checking if it starts with 'a'. Then `w[0] == 'a'` checks if 
it starts with 'a'.\n\nWait, let me re-read. The statement is `w == w.lower() and w[0] == 'a'`. The first 
computation is `w == w.lower()` which checks if the word is all lowercase (unchanged when lowercased), not if it's 
a palindrome. Then `w[0] == 'a'` checks the first letter.\n\nActually, looking at this more carefully - `w == 
w.lower()` doesn't check palindrome (that would be `w == w[::-1]`). So neither computation checks for palindrome 
correctly. But `w[0] == 'a'` does check sub-goal A.\n\nThe first part `w == w.lower()` is neither A nor B 
correctly. But `w[0] == 'a'` is sub-goal A. Since we judge the FIRST computation/statement, which is the entire `w 
== w.lower() and w[0] == 'a'` expression as one statement...\n\nThe first operation evaluated is `w == w.lower()` 
(short-circuit), which is NEITHER A nor B (it's checking case). But `w[0] == 'a'` is sub-goal A.\n\nSince the whole
statement contains `w[0] == 'a'` (sub-goal A) but not a correct implementation of B, and the first part `w == 
w.lower()` is unrelated...\n\nI'll treat the entire single statement and note it contains sub-goal A (`w[0] == 
'a'`) but not B.\n\n<verdict>A</verdict><why>The condition `w[0] == 'a'` checks if the word starts with 'a' 
(sub-goal A); the `w == w.lower()` part checks case equality, not palindrome, so sub-goal B is not pursued.</why>",
    'triggered': True
}

readout:
{
    'source': 'src',
    'dest': 'dst',
    'patched': 'src',
    'valid': True,
    'shifted': True,
    'details': {
        'source': {
            'label': 'src',
            'why': "The phrase explicitly describes checking if a string starts with 'a', which directly 
corresponds to sub-goal A.",
            'raw': "<verdict>A</verdict>\n<why>The phrase explicitly describes checking if a string starts with 
'a', which directly corresponds to sub-goal A.</why>"
        },
        'dest': {
            'label': 'dst',
            'why': 'Checking if a string equals its reverse is exactly the palindrome check, which corresponds to 
sub-goal (B): unchanged when reversed.',
            'raw': '<verdict>B</verdict><why>Checking if a string equals its reverse is exactly the palindrome 
check, which corresponds to sub-goal (B): unchanged when reversed.</why>'
        },
        'patched': {
            'label': 'src',
            'why': "The phrase explicitly describes checking if a string starts with 'a', which directly 
corresponds to sub-goal A.",
            'raw': "<verdict>A</verdict>\n<why>The phrase explicitly describes checking if a string starts with 
'a', which directly corresponds to sub-goal A.</why>"
        }
    }
}

gaze   :
{
    'src_patched': 0.2900568976998329,
    'dst_patched': 0.14096508920192719,
    'shifted': True,
    'margin': 0.1490918084979057,
    'src_clean': 0.05950939320027828,
    'dst_clean': 0.7444052416831255,
    'clean_prefers_src': False
}

---
## Cross-task multi-predicate transplant (qualitative)

SRC and DST have **different** task sets (not the shared-task `q_transfer_setups` case). SRC = {T2, T4}, T2 prefilled, about to write **T4**. DST = {T1, T2, T5}, T1 prefilled, frontier **T2**. Same pivot (trailing indent). Transplant SRC's pivot query into DST — does DST pursue T4 (SRC's target, *absent* from DST's own task) instead of its frontier T2? Eyeball only.

In [14]:
# ── cross-task transplant: SRC {T2,T4} (target T4)  ->  DST {T1,T2,T5} (frontier T2) ──
from src.attention import get_attention_matrices
bank = D.QTASKS["list[str]"]; by = {p.id: p for p in bank.predicates}
T1, T2, T3, T4, T5 = "has_palindrome", "total_length", "count_long", "count_capitalized", "distinct_count"
NAMING = "letters"

def _mk(pred_ids, progress, target):
    task = D.dataclasses.replace(bank, id="xtask", predicates=tuple(by[i] for i in pred_ids))
    return D.QTransferInstance(task=task, var_name=bank.var_names[0], presentation="listed",
        src_lang="python", dst_lang="python", src_progress=progress, dst_progress=progress,
        expected_next=target, naming=NAMING, meta={"datatype": "list[str]"})

xsrc_inst = _mk([T2, T4], [T2], T4)          # SRC: T2 done, target T4
xdst_inst = _mk([T2, T1, T5], [T1], T2)      # DST: T1 done, frontier T2
# xsrc_inst = _mk([T1, T2, T3], [T1, T2], T3)
# xdst_inst = _mk([T1, T4, T2], [T1], T4)

_q, _pf = xsrc_inst.src_parts(); xsrc_prompt = get_prompt(_q, D._finalize_prefill(_pf))
_q, _pf = xdst_inst.dst_parts(); xdst_prompt = get_prompt(_q, D._finalize_prefill(_pf))
xpiv_src, xpiv_dst = pivot_of(xsrc_prompt), pivot_of(xdst_prompt)
print(f"SRC target (T4) = {by[T4].description!r}   |   DST frontier (T2) = {by[T2].description!r}")
show(xsrc_prompt, f"SRC · {{{T2},{T4}}} · {T2} done · target {T4} · pivot@{xpiv_src}", piv=-1)
show(xdst_prompt, f"DST · {{{T1},{T2},{T5}}} · {T1} done · frontier {T2} · pivot@{xpiv_dst}", piv=-1)

# transplant SRC pivot query -> DST pivot (same machinery, inline specs)
xQ_LAYERS = sorted({l for l, _ in PATCH_HEADS})
xq_src = cache_src_proj(xsrc_prompt, [xpiv_src], xQ_LAYERS, "q")
_qh = heads_per_layer(PATCH_HEADS, "q")
xpf = [dict(which="q", cache=xq_src, heads=_qh, positions=[xpiv_dst])]   # prefill
xdc = [dict(which="q", cache=xq_src, heads=_qh, positions=[-1])]          # decode re-clamp
xsrc_tok = prepare_input(prompts=xsrc_prompt, tokenizer=mt)
xdst_tok = prepare_input(prompts=xdst_prompt, tokenizer=mt)

# ATTN over the DST TASK span (T1,T2,T5 — the SRC target T4 is NOT in DST, so it can't be shown); query = pivot
x_spans   = [token_span(xdst_prompt, p.description) for p in xdst_inst.task.predicates]
xTASK_POS = list(range(min(s[0] for s in x_spans), max(s[1] for s in x_spans)))
xtask_toks = [mt.tokenizer.decode([int(xdst_tok["input_ids"][0][j])]) for j in xTASK_POS]
x_clean_i = get_attention_matrices(xdst_tok, mt, value_weighted=True)
def _xcattn(h): return pivot_attn(x_clean_i, list(h), xTASK_POS, xpiv_dst)

# source_run — clean SRC (target T4)
rich_print(f"[red1]source_run · clean SRC (target {T4!r}):[/red1] \n{escape(greedy(xsrc_tok, max_new=96))}")
goalscope_readout("   ↳ goalscope · source_run", xsrc_prompt, xpiv_src, heads=PATCH_HEADS)

# destination_run — clean DST (frontier T2); ATTN_VIS baseline
rich_print(f"[sea_green3]destination_run · clean DST (frontier {T2!r}):[/sea_green3] \n{escape(greedy(xdst_tok, max_new=96))}")
goalscope_readout("   ↳ goalscope · destination_run", xdst_prompt, xpiv_dst, heads=PATCH_HEADS)
show_delta_tokens(xtask_toks, _xcattn(PATCH_HEADS), "ATTN_VIS · pivot→DST-TASK (clean · PATCH_HEADS)")

# patched_run — SRC query injected into DST (expect T4); ATTN_DIFF
x_iface = make_patch_interface(mt, xpf)
xgen_patched = greedy(xdst_tok, prefill_specs=xpf, decode_specs=xdc, n_steer=32, max_new=96)
rich_print(f"[salmon1]patched_run · DST (expect SRC target {T4!r}):[/salmon1] \n{escape(xgen_patched)}")
goalscope_readout("   ↳ goalscope · patched_run", xdst_prompt, xpiv_dst, heads=PATCH_HEADS, iface=x_iface)
x_patched_i = get_attention_matrices(xdst_tok, mt, value_weighted=True, patch_interface=x_iface)
show_delta_tokens(xtask_toks, pivot_attn(x_patched_i, list(PATCH_HEADS), xTASK_POS, xpiv_dst) - _xcattn(PATCH_HEADS),
                  "ATTN_DIFF · patched−clean (PATCH_HEADS)")
show_delta_tokens(xtask_toks, pivot_attn(x_patched_i, list(GOAL_HEADS), xTASK_POS, xpiv_dst),
                  "ATTN_VIS · patched (GOAL_HEADS)")

SRC target (T4) = 'the number of capitalized words'   |   DST frontier (T2) = 'the total number of characters across all words'


╭─ SRC · {total_length,count_capitalized} · total_length done · target count_capitalized · pivot@111 ─────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the total number of characters across all words\n                                                           │
│ (2) the number of capitalized words\n                                                                           │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = sum(len(w) for w in words)\n                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ DST · {has_palindrome,total_length,distinct_count} · has_palindrome done · frontier total_length · pivot@124 ──╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the total number of characters across all words\n                                                           │
│ (2) whether any word is a palindrome\n                                                                          │
│ (3) the number of distinct words\n                                                                              │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     b = any(w == w[::-1] for w in words)\n                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

source_run · clean SRC (target 'count_capitalized'): 
 b = sum(1 for w in words if w.istitle())
    return a, b
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · source_run  pos=111 '    ' => I'm checking if the character is uppercase.

... (5 outputs left out)

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · patched_run  pos=124 '    ' => I'm checking if the current element is equal to its reverse (to 
determine if it's a palindrome).

ATTN_DIFF · patched−clean (PATCH_HEADS)   (green +Δ / red −Δ, |Δ|max=1.274)

ATTN_VIS · patched (GOAL_HEADS)   (green +Δ / red −Δ, |Δ|max=1.331)